# Reference systems

The nine reference systems of Table IV, trained and evaluated under the same protocol as the
hierarchical model: a majority-class and a stratified-random predictor, TF-IDF with logistic
regression, logistic regression over frozen LaBSE embeddings, and flat twelve-class
classifiers on LaBSE, mBERT, XLM-RoBERTa, KazBERT and Kaz-RoBERTa.

It also trains models directly on the coarser targets, so that deriving a coarse decision
from the twelve-class model can be compared against optimising that decision directly.

Input: `dataset_v20.xlsx` and `gold_v20.xlsx`. All systems are trained on the authentic posts
only; the synthetic posts are excluded, as in the reported configuration.


In [ ]:
!pip install transformers torch pandas openpyxl scikit-learn -q

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, numpy as np, random, gc, json
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.utils.class_weight import compute_class_weight
from torch.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

LABEL_MAP_L1 = {"Joy":0,"Sadness":1,"Anger":2,"Fear":3,"Surprise":4,"Disgust":5,"Neutral":6,"Shame":7}
LABEL_MAP_L2 = {"Joy":0,"Hopelessness":1,"Worthlessness":2,"Exhaustion":3,"Loneliness":4,
                "Frustration":5,"Indignation":6,"Shame":7,"Fear":8,"Surprise":9,
                "Disgust":10,"Neutral":11}
L2_TO_L1 = {"Joy":"Joy","Hopelessness":"Sadness","Worthlessness":"Sadness",
            "Exhaustion":"Sadness","Loneliness":"Sadness","Frustration":"Anger",
            "Indignation":"Anger","Shame":"Shame","Fear":"Fear","Surprise":"Surprise",
            "Disgust":"Disgust","Neutral":"Neutral"}
PARENT = np.array([0,1,1,1,1,2,2,7,3,4,5,6])
RISK   = np.array([0,2,2,2,1,1,1,0,1,0,0,0])
DISTRESS = (RISK > 0).astype(int)
SEEDS = [42,123,2026]
USE_AUGMENTATION = False

In [ ]:
DATA = "/kaggle/input/datasets/tenajuro/datasetv20"
df   = pd.read_excel(f"{DATA}/dataset_v20.xlsx")
gold = pd.read_excel(f"{DATA}/gold_v20.xlsx")
for d in (df, gold):
    d.dropna(subset=["emotion_10","text"], inplace=True)
    d["label_l1"]=d["emotion_10"].map(L2_TO_L1).map(LABEL_MAP_L1)
    d["label_l2"]=d["emotion_10"].map(LABEL_MAP_L2)
assert len(set(df["id"])&set(gold["id"]))==0
real=df[~df.is_augmented].reset_index(drop=True)
aug =df[df.is_augmented].reset_index(drop=True)
y_gold=gold["label_l2"].values
print(f"train {len(df)} (authentic {len(real)}) | gold {len(gold)}")

def report_all(y_true_l2, y_pred_l2, name, extra=None):
    res={"model":name,
         "L2_macro":f1_score(y_true_l2,y_pred_l2,average="macro",zero_division=0),
         "L1_macro":f1_score(PARENT[y_true_l2],PARENT[y_pred_l2],average="macro",zero_division=0)}
    rt=RISK[y_true_l2]; rp=extra["risk"] if extra and "risk" in extra else RISK[y_pred_l2]
    dt=DISTRESS[y_true_l2]; dp=extra["distress"] if extra and "distress" in extra else DISTRESS[y_pred_l2]
    res["Risk_macro"]=f1_score(rt,rp,average="macro",zero_division=0)
    res["Distress_F1"]=f1_score(dt,dp,zero_division=0)
    res["Distress_P"]=precision_score(dt,dp,zero_division=0)
    res["Distress_R"]=recall_score(dt,dp,zero_division=0)
    return res
RESULTS=[]

In [ ]:
# ----- 1-2. Majority and stratified random baselines -----
Xtr_txt, ytr = df["text"].tolist(), df["label_l2"].values
for strat, nm in [("most_frequent","Majority class"), ("stratified","Stratified random")]:
    preds_seeds = []
    for s in SEEDS:
        dc = DummyClassifier(strategy=strat, random_state=s).fit(np.zeros((len(ytr),1)), ytr)
        preds_seeds.append(dc.predict(np.zeros((len(y_gold),1))))
    rs = [report_all(y_gold, p, nm) for p in preds_seeds]
    mean = {k: np.mean([r[k] for r in rs]) for k in rs[0] if k!="model"}
    sd   = {k: np.std([r[k] for r in rs], ddof=1) for k in rs[0] if k!="model"}
    RESULTS.append({"model":nm, **{k:f"{mean[k]:.3f}±{sd[k]:.3f}" for k in mean}})
    print(nm, {k:round(mean[k],3) for k in mean})

In [ ]:
# ===== 3. TF-IDF (word + char n-grams) + LogisticRegression =====
from sklearn.pipeline import make_union
vec = make_union(
    TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_features=50000),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=3, max_features=50000),
)
Xtr = vec.fit_transform(df["text"].astype(str))
Xte = vec.transform(gold["text"].astype(str))
print("TF-IDF features:", Xtr.shape[1])

rs = []
for s in SEEDS:
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, random_state=s)
    clf.fit(Xtr, df["label_l2"].values)
    rs.append(report_all(y_gold, clf.predict(Xte), "TF-IDF + LogReg"))
mean = {k: np.mean([r[k] for r in rs]) for k in rs[0] if k!="model"}
sd   = {k: np.std([r[k] for r in rs], ddof=1) for k in rs[0] if k!="model"}
RESULTS.append({"model":"TF-IDF + LogReg", **{k:f"{mean[k]:.3f}±{sd[k]:.3f}" for k in mean}})
print({k:round(mean[k],3) for k in mean})

In [ ]:
# ===== 4. Frozen LaBSE embeddings + LogisticRegression =====
tok = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")
enc_model = AutoModel.from_pretrained("sentence-transformers/LaBSE").to(device).eval()

def embed(texts, bs=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            e = tok(texts[i:i+bs], max_length=128, padding=True, truncation=True,
                    return_tensors="pt").to(device)
            h = enc_model(**e).last_hidden_state
            m = e["attention_mask"].unsqueeze(-1).float()
            out.append(((h*m).sum(1)/m.sum(1).clamp(min=1e-9)).cpu().numpy())
    return np.vstack(out)

Etr = embed(df["text"].astype(str).tolist())
Ete = embed(gold["text"].astype(str).tolist())
del enc_model; torch.cuda.empty_cache(); gc.collect()

rs = []
for s in SEEDS:
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=s)
    clf.fit(Etr, df["label_l2"].values)
    rs.append(report_all(y_gold, clf.predict(Ete), "Frozen LaBSE + LogReg"))
mean = {k: np.mean([r[k] for r in rs]) for k in rs[0] if k!="model"}
sd   = {k: np.std([r[k] for r in rs], ddof=1) for k in rs[0] if k!="model"}
RESULTS.append({"model":"Frozen LaBSE + LogReg", **{k:f"{mean[k]:.3f}±{sd[k]:.3f}" for k in mean}})
print({k:round(mean[k],3) for k in mean})

In [ ]:
# ----- Shared training routine for the flat encoders and the directly trained models -----
class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tok):
        self.t, self.y, self.tok = texts, labels, tok
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = self.tok(str(self.t[i]), max_length=128, padding="max_length",
                     truncation=True, return_tensors="pt")
        return {"input_ids": e["input_ids"].squeeze(),
                "attention_mask": e["attention_mask"].squeeze(),
                "y": torch.tensor(self.y[i], dtype=torch.long)}

class FlatModel(nn.Module):
    def __init__(self, ckpt, n_classes, n_frozen=4):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(ckpt)
        for p in self.encoder.embeddings.parameters(): p.requires_grad = False
        for n_, p in self.encoder.named_parameters():
            if any(f"layer.{i}." in n_ for i in range(n_frozen)): p.requires_grad = False
        self.head = nn.Sequential(nn.Linear(768,256), nn.ReLU(), nn.Dropout(0.3),
                                  nn.Linear(256, n_classes))
    def forward(self, ii, am):
        h = self.encoder(input_ids=ii, attention_mask=am).last_hidden_state
        m = am.unsqueeze(-1).float()
        return self.head((h*m).sum(1)/m.sum(1).clamp(min=1e-9))

def train_flat(ckpt, label_col, n_classes, seed, epochs=12, patience=3, bs=16):
    """label_col: name of the target-label column in df/gold."""
    set_seed(seed)
    tr_r, val_df = train_test_split(real, test_size=0.15,
                                    stratify=real["label_l2"], random_state=seed)
    train_df = pd.concat([tr_r, aug]) if USE_AUGMENTATION else tr_r
    tk = AutoTokenizer.from_pretrained(ckpt)
    mk = lambda d: DS(d["text"].tolist(), d[label_col].tolist(), tk)
    trL = torch.utils.data.DataLoader(mk(train_df), batch_size=bs, shuffle=True)
    vaL = torch.utils.data.DataLoader(mk(val_df), batch_size=bs)
    teL = torch.utils.data.DataLoader(mk(gold), batch_size=bs)

    model = FlatModel(ckpt, n_classes).to(device)
    w = torch.tensor(compute_class_weight("balanced", classes=np.arange(n_classes),
                                          y=train_df[label_col].values), dtype=torch.float).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    enc = [p for p in model.encoder.parameters() if p.requires_grad]
    opt = AdamW([{"params":enc,"lr":2e-5},{"params":model.head.parameters(),"lr":5e-4}],
                weight_decay=0.01)
    steps = len(trL)*epochs
    sch = get_linear_schedule_with_warmup(opt, steps//10, steps)
    scaler = GradScaler("cuda")

    def evaluate(loader):
        model.eval(); P, Y = [], []
        with torch.no_grad():
            for b in loader:
                ii, am = b["input_ids"].to(device), b["attention_mask"].to(device)
                with autocast("cuda"): lg = model(ii, am)
                P.extend(lg.argmax(1).cpu().numpy()); Y.extend(b["y"].numpy())
        return np.array(Y), np.array(P)

    best, best_state, wait = 0, None, 0
    for ep in range(epochs):
        model.train()
        for b in trL:
            ii, am, yy = b["input_ids"].to(device), b["attention_mask"].to(device), b["y"].to(device)
            opt.zero_grad()
            with autocast("cuda"): loss = crit(model(ii, am), yy)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        Yv, Pv = evaluate(vaL)
        f1v = f1_score(Yv, Pv, average="macro", zero_division=0)
        if f1v > best:
            best, wait = f1v, 0
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    _, Pt = evaluate(teL)
    del model; torch.cuda.empty_cache(); gc.collect()
    return Pt

def run_config(ckpt, label_col, n_classes, name, extra_key=None):
    rs = []
    for s in SEEDS:
        Pt = train_flat(ckpt, label_col, n_classes, s)
        if extra_key is None:
            rs.append(report_all(y_gold, Pt, name))
        else:
            rs.append(report_all(y_gold, np.zeros_like(y_gold), name, extra={extra_key: Pt}))
        print(f"  {name} seed {s}: done")
    mean = {k: np.mean([r[k] for r in rs]) for k in rs[0] if k!="model"}
    sd   = {k: np.std([r[k] for r in rs], ddof=1) for k in rs[0] if k!="model"}
    RESULTS.append({"model":name, **{k:f"{mean[k]:.3f}±{sd[k]:.3f}" for k in mean}})
    print(name, {k:round(mean[k],3) for k in mean})
    return rs

In [ ]:
# ----- 5. Flat 12-class LaBSE (no hierarchy) -----
_ = run_config("sentence-transformers/LaBSE", "label_l2", 12, "Flat LaBSE (12-class)")

In [ ]:
# ----- 6-7. mBERT and XLM-R (flat, 12 classes) -----
_ = run_config("bert-base-multilingual-cased", "label_l2", 12, "mBERT flat (12-class)")
_ = run_config("xlm-roberta-base", "label_l2", 12, "XLM-R flat (12-class)")

In [ ]:
# ----- 8-9. Models trained directly on the coarser targets -----
# Tests whether deriving a coarse decision from the twelve-class model costs accuracy
# relative to optimising that decision directly.
for d in (df, gold, real, aug):
    d["label_risk"]     = RISK[d["label_l2"].values]
    d["label_distress"] = DISTRESS[d["label_l2"].values]

print(">>> Directly trained binary model")
_ = run_config("sentence-transformers/LaBSE", "label_distress", 2,
               "Direct binary distress", extra_key="distress")

print(">>> Directly trained three-class model")
_ = run_config("sentence-transformers/LaBSE", "label_risk", 3,
               "Direct 3-level risk", extra_key="risk")

In [ ]:
# ----- Kazakh-specific encoders: Kaz-RoBERTa and KazBERT -----
# Included so that the choice of a multilingual encoder over a Kazakh-specific one
# is supported by a measurement rather than asserted.
# Run in the same flat configuration as mBERT and XLM-R, so the comparison is like-for-like.

KAZ_MODELS = [
    ("nur-dev/kaz-roberta-conversational", "KazRoBERTa flat"),
    ("amandyk/KazakhBERTmulti",            "KazBERT flat"),
]

for ckpt, name in KAZ_MODELS:
    try:
        print(f"\n>>> {name}  ({ckpt})")
        _ = run_config(ckpt, "label_l2", 12, name)
    except Exception as e:
        print(f"  !! {name} failed to load: {type(e).__name__}: {str(e)[:200]}")
        print("     Try another Hugging Face checkpoint (see the list below).")

# Fallback checkpoints if the ones above are unavailable:
#   "issai/rembert-sentiment-analysis-polarity-classification-kazakh"
#   "kz-transformers/kaz-roberta-conversational"
#   "nur-dev/roberta-kaz-large"
# Edit KAZ_MODELS and re-run this cell.

print("\n=== All systems ===")
print(pd.DataFrame(RESULTS).to_string(index=False))

In [ ]:
# ----- Summary table -----
tab = pd.DataFrame(RESULTS)
print(tab.to_string(index=False))
tab.to_csv("/kaggle/working/baselines_table.csv", index=False)

print("""
For comparison, the hierarchical model reported in the paper (three-seed ensemble):
  12 classes macro-F1 0.590 | 8 classes macro-F1 0.634
  seed mean 0.539 +/- 0.015 and 0.580 +/- 0.019

Averaged over seeds the hierarchical model matches but does not exceed the flat LaBSE
classifier; the paired bootstrap in the cell below gives the interval on that difference.""")


In [ ]:
import glob
import os

print("SEEDS =", SEEDS)
print("\nLooking for saved probabilities:")
print(glob.glob("/kaggle/working/**/probs_seed*.npy", recursive=True))

print("\nAll .npy files:")
for f in glob.glob("/kaggle/working/**/*.npy", recursive=True):
    print(f)

In [ ]:
# ----- Paired bootstrap of any reference system against the hierarchical model -----
# Requires probs_seed*.npy from the training notebook, in the same session or attached as a dataset.
try:
    hier = np.mean([np.load(f"/kaggle/working/probs_seed{s}.npy") for s in SEEDS], axis=0).argmax(1)
    RNG = np.random.default_rng(42)
    idx = [RNG.integers(0, len(y_gold), len(y_gold)) for _ in range(2000)]

    def cmp(pred_a, pred_b, fn, na, nb):
        d = np.array([fn(y_gold[i], pred_a[i]) - fn(y_gold[i], pred_b[i]) for i in idx])
        lo, hi = np.percentile(d, [2.5, 97.5])
        p = 2*min((d<=0).mean(), (d>=0).mean())
        sig = "significant" if lo*hi > 0 else "within CI"
        print(f"  {na} - {nb}: Δ={d.mean():+.4f} [{lo:+.4f}, {hi:+.4f}] p≈{p:.3f}  {sig}")

    f_l2 = lambda a,b: f1_score(a,b,average="macro",zero_division=0)
    print("Twelve-class comparisons (substitute the reference system's predictions):")
    print("  example: cmp(flat_preds, hier, f_l2, 'flat', 'hierarchical')")
    print("Assign the reference system's predictions to a variable and call cmp().")
except FileNotFoundError:
    print("probs_seed*.npy not found - run the training notebook in this session first.")